# 03b — Frozen cohort and missingness accounting

Produces all participant/recording denominators from the assembled dataset.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main outputs: diagnosis-stratified cohort counts and metric missingness. No manuscript denominator should be typed manually.

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("03_dataset_assembly") / "statistics")
data = read_table(OUTPUT / "03_dataset_assembly" / "paper1_analysis_dataset")
cohort = (
    data.groupby("diagnosis_analysis", dropna=False)
    .agg(
        participants=("SubjectID", "nunique"),
        logical_recordings=("logical_recording_id", "nunique"),
        primary_eligible=("primary_measurement_eligible", "sum"),
    )
    .reset_index()
)
status_columns = [column for column in data if column.endswith("_status")]
missingness = (
    data.select_dtypes(include=[np.number]).isna().mean()
    .sort_values(ascending=False).rename("missing_fraction").reset_index(names="variable")
)
save_table(cohort, TABLES, "cohort_counts")
save_table(missingness, TABLES, "numeric_missingness")
display(cohort)
display(missingness.head(30))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cohort_long = cohort.melt(
    id_vars="diagnosis_analysis",
    value_vars=["participants", "logical_recordings"],
    var_name="unit", value_name="count",
)
sns.barplot(data=cohort_long, x="diagnosis_analysis", y="count", hue="unit", ax=axes[0])
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.0f", padding=3)
axes[0].set(title="Frozen cohort denominators", xlabel="", ylabel="Count")
top_missing = missingness.head(20).sort_values("missing_fraction")
sns.barplot(data=top_missing, x="missing_fraction", y="variable", color="#E2AE4D", ax=axes[1])
axes[1].set(title="Highest numeric missingness", xlabel="Missing fraction", ylabel="")
fig.tight_layout()
save_figure(fig, FIGURES, "cohort_and_missingness_summary")
plt.show()

statistics_ready = stage_gate(
    "Cohort statistics",
    data["diagnosis_analysis"].isin(["ALS", "CONTROLS"]).all(),
    ["Non-target or missing frozen diagnoses remain in the assembled table."]
    if not data["diagnosis_analysis"].isin(["ALS", "CONTROLS"]).all() else [],
    "Run Goal 1 descriptive analysis if PASS.",
)